### Цена, клик, группы и выдача

Четыре просьбы.

1. По площади, году и району угадать цену квартиры.
2. По признакам человека угадать, кликнет ли он по рекламе. На прошлых показах клик уже записан: да или нет.
3. Разложить транзакции на группы похожих. Подписей групп в таблице нет.
4. По запросу вернуть документы от самого полезного к менее полезному.

Как называется каждая задача и какой у неё ответ: число, метка, группа без имени или порядок? Почему четвёртую нельзя свести к регрессии «предскажи номер позиции 1, 2, 3, …»: что сломается, если в выдачу встанет новый документ полезнее текущего первого?


Ответ:
1. Задача на регрессию. Ответом будет число
2. Задача на классификацию. Ответом будет метка (0 - не кликнет, 1 - кликнет)
3. На кластеризацию. Ответом будет номер кластера
4. На ранжирование. Ответом будет упорядоченный список документов. Нельзя свести к регрессии, потому что если после предсказания появится новый документ полезнее всех остальных, нужно будет переобучать модель т.к. прежние значения уже не будут работать из-за сдвига



### Три вида признаков и масштаб

Квартира: число комнат, расстояние до метро в метрах, состояние «плохое < среднее < хорошее < отличное», название района.

Какой признак числовой, какой порядковый, какой категориальный? Почему район нельзя положить в модель числами 1, 2, 3 так, будто «центр» больше «севера»? Что делают с категорией, чтобы она стала столбцами из нулей и единиц, и почему один столбец из такой группы выкидывают, если рядом уже есть свободный член?

Ближайшие соседи ищутся по евклидову расстоянию. Расстояние до метро — тысячи, число комнат — единицы. Почему разница в 5 метров начинает весить как разница в 5 комнат? Какой простой ремонт это чинит и почему среднее и стандартное отклонение для него считают только по train?


Ответ:
Число комнат, расстояние - числовые, состояние квартиры - порядковый, название района - категориальный.
Районы нельзя положить в модель числами потому что могут возникнуть бессмысленные порядки и значения. Например, это будет означать, что район 3 в 3 раза сильнее района 1, или разница между районами 1 и 2 такая же, как и между 3 и 4. Для превращения категориальных признаков в числа используют векторизацию, чаще всего one-hot encoding. Если не удалить столбец, то в one-hot encoding сумма всех стобцов одной категории будет равна 1, а поэтому будет появляться линейная зависимость между признаками и свободным членом, то есть если взять функцию y = b + w1x1 + w2x2 + w3x3, где b - свободный член

			центр(x1)	север(x2)	юг(x3)
центр		1			0			0
север		0			1			0
юг			0			0			1
то, когда модель будет искать веса и биас, получится 3 уравнения и 4 неизв, например:
Центр: b + w1 = 150
Север: b + w2 = 120
Юг: b + w3 = 110
Получается, что невозможно однозначно определить коэф.
Если же мы удаляем столбец, например, юг, то b становится базовым значением для юга и относительно него сравнивают все другие столбцы, получается 3 неизвестные и 3 уравнения, коэф определяются однозначно.
!!!!!!!!!!!!!!!!ДОПИСАТЬ!!!!!!!!!!

## 2. Как устроено обучение


### Прямая, парабола и степень 300

Один признак `x`. Семейство: `a(x) = θ0 + θ1 x + θ2 x² + … + θn xⁿ`. На картинке степень 1 проходит мимо облака, степень 2 ложится параболой, степень 300 проходит через учебные точки и между ними дико колеблется. У kNN при `k = 1` ошибка на обучающих точках тоже ноль: ближайший сосед учебной точки — она сама.

Какая из трёх степеней переобучена и по какому сравнению это видно, если кроме синих учебных точек есть зелёные, которых при подборе θ не было? Почему ноль на train совместим и с удачной простой формулой, и с зубрёжкой конечной выборки?


Ответ: Переобучена 3 модель, она очень хорошо подстроилась под учебные данные и прошла через много синих точек, но не научилась правильно находить закономерность, т.к. слишком сильно колеблется. Это видно по сравнению ошибок на тренировочных и тестовых данных. На тренировочных ошибка будет почти нулевая, а на тестовой намного выше. Ноль на train совместим и с удачной простой формулой, и с зубрежкой конечной выборки т.к. нулевая ошибка на трейн говорит только о том, что модель идеально предсказала уже известные ей данные, а она могла либо найти закономерность, либо, если произошло переобучение зазубрить трейн.

### Train, validation, test

Выборку разделили на три части. Веса θ подбирают на первой. На второй после каждого круга решают, менять ли семейство и гиперпараметры: перебрали `k` от 10 до 5 и оставили 5, потому что там ошибка меньше. Третью часть смотрят один раз в конце.

Как называется каждая часть? Почему выбранное `k = 5` модель фактически узнала со второй части, хотя веса на ней не учили? Зачем тогда третья, если вторая уже ловит переобучение?


Ответ: Первая часть - train, вторая - validation, третья - test. k = 5 мы подобрали, потому что модель выдала наименьшую ошибку при таком гиперпараметре. 3 часть test необходима для независимой итоговой оценки на данных, которые она еще не видела.

### Лосс и метрика

Лосс — число от истинного ответа и предсказания: чем больше, тем хуже. Эмпирический риск — средний лосс по обучающей выборке. Обучение — выбрать θ, на котором этот средний лосс меньше.

Метрика тоже сравнивает предсказания с ответами, но веса под неё не подбирают.

Почему в регрессии MSE часто бывает и лоссом, и метрикой, а в классификации так почти никогда: лосс считают от вероятности, accuracy — уже от выбранной метки? Кому из двоих нужно, чтобы по числу было удобно искать минимум, а кому — чтобы число можно было объяснить заказчику?


Ответ: MSE в регрессии часто бывает и лоссом, и метрикой, т.к. одновременно подходит для оптимизацици и оценки. В классификации используют log loss, т.к. этот показатель честнее штрафует модель за неуверенный ответ. Лоссу - чтобы удобно искать минимум, метрике - чтобы объяснить заказчику.

## 3. Среднее, разброс, связь


### Типичная зарплата

Суммы уже посчитаны.

| Набор | Числа | Среднее | Медиана | Мода |
| --- | --- | --- | --- | --- |
| Зарплаты, тыс. | 40, 42, 45, 45, 48, 50, 52, 90 | 51.5 | 46.5 | 45 |
| Оценки | 3, 4, 4, 5, 4 | 4 | 4 | 4 |

Нужна одна цифра «типичная зарплата» для обычного сотрудника. Какую называешь и почему 51.5 описывает не его? В каком ряду среднее, медиана и мода не спорят?

Константный прогноз «всем одну цену» оптимален в разных точках. Для MSE эта точка — среднее, для MAE — медиана. Как это видно на ряде зарплат: куда 90 утаскивает среднее и где остаётся медиана?


Ответ: Назову медиану - 46.5. 51.5 не описывает правильно типичную зарплату, т.к. в выборке наблюдается выброс - 90, из-за него только у 2 человек из 8 зарплата больше заявленной средней. MSE возводит в квадрат, поэтому она сильно штрафует модель за большую ошибку в выбросе. 90 вытягивает среднее вправо. MAE учитывает линейно, поэтому если выбросов немного, то медиана остается устойчивой

### Свойства среднего и дисперсии

Дисперсия здесь — сумма квадратов отклонений от своего среднего, делённая на n. Стандартное отклонение — корень из неё.

1. К каждой зарплате прибавили 10 тысяч премии. Куда сдвинется среднее? Почему дисперсия не изменится?
2. Каждую зарплату умножили на 2. Что станет со средним? Почему дисперсия вырастет в 4 раза, а стандартное отклонение только в 2?
